# 🚗 Used Car Price Predictor — Google Colab

**How to use:**
1. Upload all 4 CSV files to Colab (Files panel on left)
2. Run each cell in order
3. Paste ngrok token in Cell 7 → get your live URL

> Files needed: `car_data.csv`, `CAR_DETAILS_FROM_CAR_DEKHO.csv`, `Car_details_v3.csv`, `car_details_v4.csv`

In [ ]:
!pip install -q streamlit pyngrok
print('Done!')

## Cell 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings, pickle
warnings.filterwarnings('ignore')
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
print('Imports OK')

## Cell 3 — Load All 4 Datasets

In [ ]:
def extract_num(series):
    import re
    return pd.to_numeric(series.astype(str).str.extract(r'([\d.]+)')[0], errors='coerce')

# Dataset 1: car_data.csv
d1 = pd.read_csv('car_data.csv')
d1 = d1.rename(columns={'Car_Name':'name','Year':'year','Selling_Price':'selling_price_lakh',
    'Kms_Driven':'km_driven','Fuel_Type':'fuel','Seller_Type':'seller_type',
    'Transmission':'transmission','Owner':'owner'})
d1['selling_price'] = d1['selling_price_lakh'] * 1e5
d1['brand'] = d1['name'].str.split().str[0]
d1['mileage_kmpl'] = np.nan; d1['engine_cc'] = np.nan
d1['max_power_bhp'] = np.nan; d1['seats'] = 5.0
d1['owner'] = d1['owner'].map({0:'First Owner',1:'Second Owner',
    2:'Third Owner',3:'Fourth & Above Owner'}).fillna('First Owner')

# Dataset 2: CAR_DETAILS_FROM_CAR_DEKHO.csv
d2 = pd.read_csv('CAR_DETAILS_FROM_CAR_DEKHO.csv')
d2['brand'] = d2['name'].str.split().str[0]
d2['mileage_kmpl'] = np.nan; d2['engine_cc'] = np.nan
d2['max_power_bhp'] = np.nan; d2['seats'] = 5.0

# Dataset 3: Car_details_v3.csv
d3 = pd.read_csv('Car_details_v3.csv')
d3['brand'] = d3['name'].str.split().str[0]
d3['mileage_kmpl'] = extract_num(d3['mileage'])
d3['engine_cc'] = extract_num(d3['engine'])
d3['max_power_bhp'] = extract_num(d3['max_power'])
d3['seats'] = d3['seats'].fillna(5.0)

# Dataset 4: car_details_v4.csv
d4 = pd.read_csv('car_details_v4.csv')
d4 = d4.rename(columns={'Make':'brand','Price':'selling_price','Year':'year',
    'Kilometer':'km_driven','Fuel Type':'fuel','Transmission':'transmission',
    'Owner':'owner','Seller Type':'seller_type','Seating Capacity':'seats'})
d4['name'] = d4['brand'] + ' ' + d4['Model'].astype(str)
d4['engine_cc'] = extract_num(d4['Engine'])
d4['max_power_bhp'] = extract_num(d4['Max Power'])
d4['mileage_kmpl'] = np.nan
d4['owner'] = d4['owner'].astype(str).replace({'First':'First Owner','Second':'Second Owner',
    'Third':'Third Owner','Fourth':'Fourth & Above Owner','Fourth & Above':'Fourth & Above Owner'})

KEEP = ['name','brand','year','selling_price','km_driven','fuel','transmission',
        'owner','seller_type','mileage_kmpl','engine_cc','max_power_bhp','seats']
combined = pd.concat([d1[KEEP], d2[KEEP], d3[KEEP], d4[KEEP]], ignore_index=True)
print(f'Combined: {combined.shape[0]:,} rows from 4 datasets')

## Cell 4 — Clean & Feature Engineering

In [ ]:
for c in ['fuel','transmission','owner','seller_type']:
    combined[c] = combined[c].astype(str).str.strip()

combined['fuel'] = combined['fuel'].replace({'Cng':'CNG','Lpg':'LPG',
    'Cng + Cng':'CNG','Petrol + Cng':'CNG','Petrol + Lpg':'LPG'})

combined = combined[combined['fuel'].isin(['Petrol','Diesel','CNG','LPG','Electric'])]
combined = combined[combined['owner'].isin(['First Owner','Second Owner','Third Owner',
    'Fourth & Above Owner','Test Drive Car'])]
combined = combined.dropna(subset=['selling_price','year','km_driven','fuel','transmission','owner'])
combined = combined[combined['selling_price'] > 5000]
combined = combined[combined['year'].between(1990, 2024)]

mu, sd = combined['selling_price'].mean(), combined['selling_price'].std()
combined = combined[combined['selling_price'].between(mu - 3*sd, mu + 3*sd)]

combined['car_age'] = 2024 - combined['year']  # hint from problem statement
combined['seats'] = combined['seats'].fillna(5.0)

print(f'Clean dataset: {combined.shape[0]:,} rows')
print(f'Brands : {combined["brand"].nunique()}')
print(f'Fuels  : {sorted(combined["fuel"].unique())}')

## Cell 5 — Encode & Train Model

In [ ]:
cat_cols = ['fuel','transmission','owner','seller_type','brand']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined[col+'_enc'] = le.fit_transform(combined[col].astype(str))
    encoders[col] = le

FEATURES = ['car_age','km_driven','fuel_enc','transmission_enc','owner_enc',
            'seller_type_enc','brand_enc','mileage_kmpl','engine_cc','max_power_bhp','seats']

df_m = combined[FEATURES + ['selling_price']].copy()
for col in ['mileage_kmpl','engine_cc','max_power_bhp']:
    df_m[col] = df_m[col].fillna(df_m[col].median())
df_m = df_m.dropna()
X, y = df_m[FEATURES], df_m['selling_price']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
m_eval = GradientBoostingRegressor(n_estimators=300, learning_rate=0.08, max_depth=5, subsample=0.85, random_state=42)
m_eval.fit(X_tr, y_tr)
print(f'R2: {r2_score(y_te, m_eval.predict(X_te)):.4f}  RMSE: Rs {np.sqrt(mean_squared_error(y_te, m_eval.predict(X_te))):,.0f}')

model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.08, max_depth=5, subsample=0.85, random_state=42)
model.fit(X, y)

META = {
    'brands':         sorted(combined['brand'].unique().tolist()),
    'fuels':          sorted(combined['fuel'].unique().tolist()),
    'transmissions':  sorted(combined['transmission'].unique().tolist()),
    'owners':         sorted(combined['owner'].unique().tolist()),
    'seller_types':   sorted(combined['seller_type'].unique().tolist()),
    'year_min':       int(combined['year'].min()),
    'year_max':       int(combined['year'].max()),
    'mileage_median': float(df_m['mileage_kmpl'].median()),
    'engine_median':  float(df_m['engine_cc'].median()),
    'power_median':   float(df_m['max_power_bhp'].median()),
}
print(f'Model trained on {len(X):,} rows | {len(META["brands"])} brands')

## Cell 6 — Write app.py & Save Artifacts

In [ ]:
# Build app.py line by line to avoid string conflicts
lines = []
lines.append('import streamlit as st')
lines.append('import pandas as pd')
lines.append('import numpy as np')
lines.append('import pickle')
lines.append('')
lines.append('st.set_page_config(page_title="Used Car Price Predictor", page_icon="car", layout="centered")')
lines.append('')
lines.append('CSS = """')
lines.append('<style>')
lines.append('@import url(\"https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap\");')
lines.append('html,body,[class*="css"]{font-family:Inter,sans-serif}')
lines.append('#MainMenu,footer,header{visibility:hidden}')
lines.append('.block-container{padding:2rem 2rem 4rem;max-width:780px}')
lines.append('.result-box{background:linear-gradient(135deg,#0f2d1f,#1a3a2a);border:1.5px solid #2ecc71;border-radius:14px;padding:30px 36px;text-align:center;margin:12px 0}')
lines.append('.result-label{font-size:12px;font-weight:600;color:#7dbf7d;letter-spacing:2px;text-transform:uppercase;margin-bottom:8px}')
lines.append('.result-price{font-size:52px;font-weight:700;color:#2ecc71;line-height:1.1}')
lines.append('.result-range{font-size:13px;color:#5a9e5a;margin-top:10px}')
lines.append('.section-label{font-size:11px;font-weight:700;letter-spacing:2px;color:#888;text-transform:uppercase;margin-top:20px;margin-bottom:2px}')
lines.append('</style>')
lines.append('"""')
lines.append('st.markdown(CSS, unsafe_allow_html=True)')
lines.append('')
lines.append('@st.cache_resource')
lines.append('def load_artifacts():')
lines.append('    with open("model.pkl","rb") as f: obj = pickle.load(f)')
lines.append('    with open("meta.pkl","rb") as f: meta = pickle.load(f)')
lines.append('    return obj["model"],obj["encoders"],obj["features"],meta')
lines.append('')
lines.append('model,encoders,FEATURES,meta = load_artifacts()')
lines.append('')
lines.append('def enc(col,val):')
lines.append('    le=encoders[col]')
lines.append('    return int(le.transform([val])[0]) if val in le.classes_ else 0')
lines.append('')
lines.append('def predict(brand,year,km,fuel,transmission,owner,seller_type,mileage,engine_cc,max_power,seats):')
lines.append('    car_age = 2024 - year')
lines.append('    row = pd.DataFrame([[car_age,km,enc("fuel",fuel),enc("transmission",transmission),')
lines.append('        enc("owner",owner),enc("seller_type",seller_type),enc("brand",brand),')
lines.append('        mileage,engine_cc,max_power,float(seats)]],columns=FEATURES)')
lines.append('    return max(float(model.predict(row)[0]),10000)')
lines.append('')
lines.append('st.title("Used Car Price Predictor")')
lines.append('st.caption("Trained on 14,475 listings — car_data, CarDekho, v3, v4")')
lines.append('st.divider()')
lines.append('')
lines.append('st.markdown("<p class=section-label>Car Details</p>",unsafe_allow_html=True)')
lines.append('col1,col2 = st.columns(2)')
lines.append('with col1: brand = st.selectbox("Brand",options=sorted(meta["brands"]))')
lines.append('with col2: year = st.number_input("Year",min_value=meta["year_min"],max_value=2024,value=2019,step=1)')
lines.append('km = st.number_input("Kilometres Driven",min_value=0,max_value=600000,value=45000,step=1000,format="%d")')
lines.append('')
lines.append('st.markdown("<p class=section-label>Specifications</p>",unsafe_allow_html=True)')
lines.append('c3,c4=st.columns(2)')
lines.append('with c3: fuel=st.selectbox("Fuel Type",options=meta["fuels"])')
lines.append('with c4: transmission=st.selectbox("Transmission",options=meta["transmissions"])')
lines.append('c5,c6,c7=st.columns(3)')
lines.append('with c5: mileage=st.number_input("Mileage (kmpl)",min_value=0.0,max_value=60.0,value=round(meta["mileage_median"],1),step=0.1)')
lines.append('with c6: engine_cc=st.number_input("Engine (CC)",min_value=500,max_value=6000,value=int(meta["engine_median"]),step=50)')
lines.append('with c7: max_power=st.number_input("Max Power (bhp)",min_value=20.0,max_value=700.0,value=round(meta["power_median"],1),step=1.0)')
lines.append('seats=st.select_slider("Seats",options=[2,4,5,6,7,8,9],value=5)')
lines.append('')
lines.append('st.markdown("<p class=section-label>Ownership and Seller</p>",unsafe_allow_html=True)')
lines.append('c8,c9=st.columns(2)')
lines.append('with c8: owner=st.selectbox("Owner Type",options=meta["owners"])')
lines.append('with c9: seller_type=st.selectbox("Seller Type",options=meta["seller_types"])')
lines.append('')
lines.append('if st.button("Predict Resale Price",use_container_width=True,type="primary"):')
lines.append('    with st.spinner("Running model..."):')
lines.append('        price=predict(brand,year,km,fuel,transmission,owner,seller_type,mileage,engine_cc,max_power,seats)')
lines.append('    pl=price/1e5; lo=pl*0.9; hi=pl*1.1')
lines.append('    st.markdown(f"""<div class=result-box><div class=result-label>Estimated Resale Value</div>')
lines.append('        <div class=result-price>Rs {pl:.2f} L</div>')
lines.append('        <div class=result-range>Range: Rs {lo:.2f}L to Rs {hi:.2f}L</div></div>""",unsafe_allow_html=True)')
lines.append('    with st.expander("Summary",expanded=True):')
lines.append('        a,b,c,d=st.columns(4)')
lines.append('        a.metric("Car Age",f"{2024-year} yrs")')
lines.append('        b.metric("KMs",f"{km:,}")')
lines.append('        c.metric("Fuel",fuel)')
lines.append('        d.metric("Price",f"Rs {pl:.2f}L")')

with open('app.py','w') as f:
    f.write('\n'.join(lines))

with open('model.pkl','wb') as f:
    pickle.dump({'model':model,'encoders':encoders,'features':FEATURES},f)
with open('meta.pkl','wb') as f:
    pickle.dump(META,f)

print('app.py, model.pkl, meta.pkl saved in Colab session!')

## Cell 7 — Launch Streamlit via ngrok

Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = 'your_ngrok_authtoken_here'  # <-- paste your token here

ngrok.set_auth_token(NGROK_TOKEN)

proc = subprocess.Popen(
    ['streamlit','run','app.py','--server.port=8501','--server.headless=true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)
tunnel = ngrok.connect(8501)
print('App is live at:', tunnel.public_url)
print('Open this link in any browser!')